# 01 — Quickstart: Structured Output

**Stage 1 of the workshop.** First script, first proof that Strands turns an LLM into something with a typed contract instead of loose text.


## Problem

An LLM's native output is unstructured text. If your application needs a `name`, an `age`, and a `job title` as real typed fields — not a paragraph you regex out afterward — you're stuck writing brittle parsing code that breaks the moment the model phrases things differently.

The problem this notebook solves: **get validated, typed data back from an agent call, not prose you have to parse.**


## Concept

The abstraction is `structured_output_model` — pass a Pydantic model as a schema, and the agent constrains its response to match it. The model is *forced* into the schema, not *hoping* its free-text answer happens to match a pattern you wrote after the fact.

Three moving parts, same as every Strands script in this workshop:
- **Model** — `model_provider.get_model()` (Ollama first, Bedrock fallback)
- **Agent** — `Agent(model=model)`, no tools needed for this example
- **Schema** — a plain `pydantic.BaseModel` describing the shape you want back

**When would you NOT use this?** When you genuinely want free-text prose (a summary, an essay) — structured output is for when the *caller* needs typed fields, not readable paragraphs.


## Architecture

```
                         ┌──────────────────┐
   "John Smith is a      │                  │
    30-year-old software │  Agent (model)   │
    engineer"    ───────▶│                  │
                         └────────┬─────────┘
                                  │ constrained by
                                  ▼
                         ┌──────────────────┐
                         │  PersonInfo       │
                         │  (Pydantic model) │
                         │  - name: str      │
                         │  - age: int       │
                         │  - occupation: str│
                         └────────┬─────────┘
                                  │
                                  ▼
                    result.structured_output
                    (a real PersonInfo instance,
                     not a string to parse)
```

No tool calls, no loop branching — this is the simplest possible agent invocation, just with a schema attached.


## Step 1 — Resolve the model

Same resolver every script in this workshop uses: Ollama first, Bedrock fallback. See `workshop/model_provider.py`.


In [1]:
import sys
from pathlib import Path

# Two levels up from this notebook (01-quickstart/) lands at workshop/,
# where model_provider.py lives — same sys.path pattern the .py scripts use.
sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from pydantic import BaseModel
from strands import Agent

model = get_model()
print(f"Using: {type(model).__name__}")


Using: OllamaModel


## Step 2 — Define the schema

`PersonInfo` is the contract. The agent's output will be forced to match this shape — three required fields, three real Python types.


In [2]:
class PersonInfo(BaseModel):
    name: str
    age: int
    occupation: str


## Step 3 — Run the agent with `structured_output_model`

This is the whole trick: pass the schema class as `structured_output_model`, and `result.structured_output` comes back as a real `PersonInfo` instance — not a string you'd have to parse yourself.


In [5]:
agent = Agent(model=model)
result = agent(
    "John Smith is a 30-year-old software engineer",
    structured_output_model=PersonInfo,
)



Tool #1: PersonInfo


## Step 4 — Use the typed result

No regex, no `.split()`, no hoping the model's phrasing matches a pattern — just attribute access on a validated object.


In [6]:
print(f"Name: {result.structured_output.name}")
print(f"Age: {result.structured_output.age}")
print(f"Job: {result.structured_output.occupation}")

Name: John Smith
Age: 30
Job: software engineer


## Failure mode to know about

If the input text doesn't actually contain the fields your schema demands, the agent either fails validation or has to guess/hallucinate a value to satisfy the required field — structured output constrains *shape*, not *truthfulness*. Always sanity-check outputs against source data for anything that matters.
